# Water in Water: Implicit Solvation

:::{div}
:class: colab-badge
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blinkletter/UsingPsi4v2/blob/main/temp/14-solvation.ipynb)
:::


In [1]:
####################################################
### Import Libraries and set up global variables ###
####################################################

from pathlib import Path
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Psi4 cannot be installed with pip. On Colab we first install conda
# (condacolab), which RESTARTS the session. After the restart, run
# this cell again and Psi4 will be installed with conda.
try:
    import psi4
except ImportError:
    if IN_COLAB:
        !pip -q install condacolab
        import condacolab
        condacolab.install()
        !mamba install -q -y -c conda-forge psi4
    import psi4

try:
    import py3Dmol
except ImportError:
    !pip -q install py3Dmol
    import py3Dmol

# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Style sheets and helper functions from the book's GitHub repository
if IN_COLAB:
    if not Path("styles/tufte.mplstyle").exists():
        !wget -q -N -P styles https://raw.githubusercontent.com/blinkletter/UsingPsi4v2/main/styles/tufte.mplstyle
        !wget -q -N https://raw.githubusercontent.com/blinkletter/UsingPsi4v2/main/psi4_helpers.py
    plt.style.use("styles/tufte.mplstyle")
    nproc = 2          # Colab gives you 2 cores
else:
    sys.path.insert(0, "..")                    # psi4_helpers.py is one folder up
    plt.style.use("../styles/tufte.mplstyle")
    nproc = 4          # house limit: 4 threads

import psi4_helpers as ph

# All Psi4 .out and .log files go into the output folder
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

# Resources: house limits are 4 threads and 4 GB
psi4.set_num_threads(nproc)
_ = psi4.set_memory("2 GB")


  Threads set to 4 by Python driver.



  Memory set to   1.863 GiB by Python driver.


Every calculation so far has been in the gas phase. Chemistry usually happens in solution. The cheapest way to include a solvent is an **implicit** (continuum) model: the solute sits in a molecule-shaped **cavity** inside a featureless medium with the dielectric constant of the solvent ($\varepsilon = 78.4$ for water). The solute's charge distribution polarizes the medium, and the medium's reaction field polarizes the solute back. *Psi4* does this with the **polarizable continuum model** (PCM).

## What You Will Learn
- How to switch on PCM in *Psi4* and describe the solvent and cavity.
- How solvation changes the **dipole moment** of water.
- How large solvation energies are for neutral molecules and for **ions**.
- How solvation changes the energy of a hydrogen bond and of the **autoionization** of water.
- The limits of a continuum model.

Experimental hydration free energy of water: $\Delta G_{\mathrm{solv}} = -26.4$ kJ/mol (Ben-Naim and Marcus, *J. Chem. Phys.* **1984**, *81*, 2016).

In [2]:
psi4.set_output_file("output/14-solvation.out", False);
h2kJ = psi4.constants.hartree2kJmol
au2debye = psi4.constants.dipmom_au2debye

import contextlib

## 1. Gas-Phase Structures

PCM in *Psi4* has no analytic gradients, so optimizing in solution would need many finite-difference steps. We optimize in the gas phase and then calculate single-point energies with and without PCM. Water and its ions change little in structure on solvation, so this is a common and reasonable shortcut.

```{note} Psi4 settings
:class: dropdown
`b3lyp/6-31+g*`
: The same level used for the ions in the *Acids and Bases* notebook; the diffuse functions are needed for $\mathrm{OH^-}$. *Alternatives:* `hf/6-31+g*` (the original level used to parameterize many PCM radii), `wb97x-d/def2-tzvpd`.

`symmetry c1`, `no_reorient`, `no_com`
: The PCMSolver library cannot use point-group symmetry, so we switch symmetry off. Keeping the orientation fixed avoids rebuilding the cavity in a new frame.
```

In [3]:
psi4.set_options({"basis": "6-31+g*", "scf_type": "pk"})

def make(geom):
    mol = psi4.geometry(geom + "\nsymmetry c1\nno_reorient\nno_com\n")
    return mol

molecules = {
    "H2O":  make("0 1\nO\nH 1 0.96\nH 1 0.96 2 104.5"),
    "H3O+": make("1 1\nO 0.000 0.000 0.100\nH 0.000 0.950 -0.250\nH 0.823 -0.475 -0.250\nH -0.823 -0.475 -0.250"),
    "OH-":  make("-1 1\nO\nH 1 0.97"),
    "dimer": make("""0 1
O  -1.551007  -0.114520   0.000000
H  -1.934259   0.762503   0.000000
H  -0.599677   0.040712   0.000000
--
0 1
O   1.350625   0.111469   0.000000
H   1.680398  -0.373741  -0.758561
H   1.680398  -0.373741   0.758561"""),
}

gas = {}
dipole_gas = None
for name, mol in molecules.items():
    psi4.optimize("b3lyp", molecule=mol)
    E, w = psi4.energy("b3lyp", molecule=mol, return_wfn=True)
    gas[name] = E
    if name == "H2O":
        psi4.oeprop(w, "DIPOLE", title="GAS")
        dipole_gas = np.linalg.norm(w.variable("GAS DIPOLE")) * au2debye
print({k: round(v, 6) for k, v in gas.items()})

Optimizer: Optimization complete!


Optimizer: Optimization complete!


Optimizer: Optimization complete!


Optimizer: Optimization complete!


{'H2O': -76.422573, 'H3O+': -76.691526, 'OH-': -75.79668, 'dimer': -152.855413}


## 2. Switching on PCM

The PCM model is described in a short text block written in the PCMSolver input language, passed to `psi4.pcm_helper()`.

```{note} PCM settings
:class: dropdown
`pcm: True` and `pcm_scf_type: "total"`
: Turn on PCM for the SCF. `"total"` (the default) treats the nuclear and electronic potentials together; `"separate"` is mainly for debugging.

`SolverType = IEFPCM`
: The integral-equation-formalism PCM, the standard choice. *Alternative:* `CPCM` (conductor-like PCM), which treats the solvent as a perfect conductor and scales the result; it is very similar for high-dielectric solvents like water and more robust for some systems.

`Solvent = Water`
: Uses the built-in dielectric constant for water. Other solvents can be named (`Methanol`, `Acetonitrile`, `Toluene`, ...), or a custom `Epsilon` can be given.

The cavity: `Type = GePol`, `RadiiSet = Bondi`, `Scaling = True`, `Area = 0.3`
: The cavity is built from interlocking spheres on each atom. `Bondi` radii are the classic van der Waals radii; `Scaling = True` multiplies them by 1.2, the usual choice. *Alternative:* `UFF` radii. `Area` is the size (in Å²) of the small surface patches (tesserae) the surface is divided into; smaller means more accurate and slower.

Where the extra files go
: PCMSolver writes `PEDRA.OUT` and `cavity` files into the current folder. The helper below temporarily changes into the `output` folder so they land there too.

The `ddx` alternative
: *Psi4* also has the newer **ddx** library (`"ddx": True`, `"ddx_model": "pcm"`, `"ddx_solvent": "water"`, `"ddx_radii_set": "uff"`), which supports COSMO, PCM and a linearized Poisson–Boltzmann model for salt solutions. It needs the `pyddx` package (`mamba install -c conda-forge pyddx`).
```

In [4]:
pcm_input = """
Units = Angstrom
Medium {
  SolverType = IEFPCM
  Solvent = Water
}
Cavity {
  RadiiSet = Bondi
  Type = GePol
  Scaling = True
  Area = 0.3
  Mode = Implicit
}
"""

def pcm_energy(mol, pcm_text=pcm_input, return_wfn=False):
    """B3LYP energy in PCM water. PCMSolver's scratch files are written into the output folder."""
    psi4.set_options({"pcm": True, "pcm_scf_type": "total"})
    psi4.pcm_helper(pcm_text)
    with contextlib.chdir(output_dir):
        result = psi4.energy("b3lyp", molecule=mol, return_wfn=return_wfn)
    psi4.set_options({"pcm": False})
    return result

## 3. Solvating Water

In [5]:
E_pcm, w_pcm = pcm_energy(molecules["H2O"], return_wfn=True)
psi4.oeprop(w_pcm, "DIPOLE", title="PCM")
dipole_pcm = np.linalg.norm(w_pcm.variable("PCM DIPOLE")) * au2debye

print(f"Solvation energy of water: {(E_pcm - gas['H2O']) * h2kJ:.1f} kJ/mol   (experiment ΔG = -26.4 kJ/mol)")
print(f"Dipole moment: gas {dipole_gas:.2f} D  →  PCM water {dipole_pcm:.2f} D")

Solvation energy of water: -28.2 kJ/mol   (experiment ΔG = -26.4 kJ/mol)
Dipole moment: gas 2.25 D  →  PCM water 2.55 D


The continuum pulls more charge onto the oxygen and the dipole moment grows by about 15 %. Liquid water molecules are known to be strongly polarized (their dipole in the liquid is estimated at 2.5–3 D), so the direction is right, although a continuum model gives only part of the effect.

The PCM energy is only the **electrostatic** part of the solvation free energy. Real solvation also involves making a cavity in the solvent, dispersion and repulsion with the solvent molecules, and specific hydrogen bonds. Agreement with experiment for water is partly a result of how the radii were chosen.

## 4. The Cavity Matters

Let us see how much the answer depends on the choice of atomic radii.

In [6]:
rows = []
# Allinger radii with Area = 0.3 give a PCMSolver warning (see below), so that cavity uses a finer mesh
for radii, scaling, area in [("Bondi", "True", "0.3"), ("UFF", "True", "0.3"),
                             ("UFF", "False", "0.3"), ("Allinger", "True", "0.2")]:
    text = (pcm_input.replace("RadiiSet = Bondi", f"RadiiSet = {radii}")
                     .replace("Scaling = True", f"Scaling = {scaling}")
                     .replace("Area = 0.3", f"Area = {area}"))
    E = pcm_energy(molecules["H2O"], pcm_text=text)
    rows.append({"radii": radii, "scaled by 1.2": scaling, "area (Å²)": area,
                 "ΔE solv (kJ/mol)": (E - gas["H2O"]) * h2kJ})
pd.DataFrame(rows).round(1)

,radii,scaled by 1.2,area (Å²),ΔE solv (kJ/mol)
0,Bondi,True,0.3,-28.2
1,UFF,True,0.3,-17.6
2,UFF,False,0.3,-29.6
3,Allinger,True,0.2,-23.9


A smaller cavity puts the solvent closer to the charges and gives a larger solvation energy. (With Allinger radii and the `Area = 0.3` used for the others, PCMSolver prints a warning that the `S matrix is not positive-definite`: the cavity surface has been cut into some awkwardly small patches. A slightly finer mesh, `Area = 0.2`, removes the warning and changes the energy by only 0.02 kJ/mol. If you see this warning, try a slightly different `Area`.) Differences of 10 kJ/mol or more between reasonable choices are common. Continuum models are best used for **differences** (comparing similar molecules with the same settings), not for absolute values.

## 5. Ions and the Autoionization of Water

Ions are solvated far more strongly than neutral molecules. We calculate the electrostatic solvation energy of the hydronium and hydroxide ions, then the energy of

$$
\mathrm{2\,H_2O \rightarrow H_3O^+ + OH^-}
$$

in the gas phase and in PCM water.

In [7]:
pcm = {"H2O": E_pcm}
for name in ["H3O+", "OH-", "dimer"]:
    pcm[name] = pcm_energy(molecules[name])

for name in ["H2O", "H3O+", "OH-"]:
    print(f"{name:5s} solvation energy: {(pcm[name] - gas[name]) * h2kJ:8.1f} kJ/mol")

dE_gas = (gas["H3O+"] + gas["OH-"] - 2 * gas["H2O"]) * h2kJ
dE_pcm = (pcm["H3O+"] + pcm["OH-"] - 2 * pcm["H2O"]) * h2kJ
print(f"\n2 H2O -> H3O+ + OH-   gas: {dE_gas:7.1f} kJ/mol    PCM water: {dE_pcm:7.1f} kJ/mol")
print("Experiment in water: ΔG ≈ +80 kJ/mol (pKw = 14)")

H2O   solvation energy:    -28.2 kJ/mol
H3O+  solvation energy:   -351.5 kJ/mol
OH-   solvation energy:   -363.3 kJ/mol

2 H2O -> H3O+ + OH-   gas:   937.1 kJ/mol    PCM water:   278.8 kJ/mol
Experiment in water: ΔG ≈ +80 kJ/mol (pKw = 14)


In the gas phase, separating a cation from an anion costs about 900 kJ/mol. In PCM water most of that cost is paid back by solvating the ions. The remaining value is still too high compared with experiment: we have left out thermal corrections, the standard-state corrections, and above all the **specific hydrogen bonds** between the ions and their first shell of water molecules, which a continuum cannot describe. A common improvement is a **cluster–continuum** model, in which a few explicit water molecules are placed around each ion and the whole cluster is embedded in PCM.

## 6. The Hydrogen Bond in Water

In [8]:
IE_gas = (gas["dimer"] - 2 * gas["H2O"]) * h2kJ
IE_pcm = (pcm["dimer"] - 2 * pcm["H2O"]) * h2kJ
print(f"Dimer binding energy (B3LYP/6-31+G*, no CP correction): gas {IE_gas:.1f} kJ/mol,  PCM {IE_pcm:.1f} kJ/mol")

Dimer binding energy (B3LYP/6-31+G*, no CP correction): gas -27.0 kJ/mol,  PCM -17.2 kJ/mol


The continuum weakens the hydrogen bond by more than a third. Each separated water molecule is well solvated by the dielectric, while in the dimer part of each molecule's surface is buried against its partner and the dielectric can no longer stabilize it. This is why hydrogen bonds that are strong in the gas phase are much weaker in water, where the solvent offers hydrogen bonds of its own.

```{important} Limits of implicit solvation
- No **specific** interactions: no hydrogen bonds to the solvent, no solvent structure.
- Results depend on the **cavity** (radii, scaling).
- Only the **electrostatic** part is calculated here; models such as SMD add non-electrostatic terms but are not available in *Psi4*.
- Gas-phase structures were used; in reality the structure relaxes a little in solution.
```

## Summary
- `pcm: True` and `psi4.pcm_helper()` place a molecule in a dielectric continuum (`symmetry c1` is required).
- Solvation polarizes water and increases its dipole moment.
- Ion solvation energies are hundreds of kJ/mol and transform gas-phase acid–base energetics.
- Continuum models are best for comparing similar species with identical settings.